In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
from pathlib import Path
import os

# Repo-relative data location (was Colab's interactive files.upload()).
DATA_DIR = Path(os.environ.get(
    'CRITMIN_DATA',
    Path.cwd().parent / 'data' / 'raw' / 'import_data',
))


In [ ]:
file_path = DATA_DIR / 'LFP-CAM.csv'

df = pd.read_csv(file_path, encoding="latin1")
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
df = df[["refYear", "partnerDesc", "cmdDesc", "primaryValue"]].copy()
df.head()

In [ ]:
df["refYear"] = pd.to_numeric(df["refYear"], errors="coerce")
df["primaryValue"] = pd.to_numeric(df["primaryValue"], errors="coerce")
df.dtypes

In [ ]:
output_folder = "/content/commodity_plots"
os.makedirs(output_folder, exist_ok=True)
print("Folder created:", output_folder)

In [ ]:
df["cmdDesc"].unique()

In [ ]:
commodity = "LFPCAM"
df_cmd = df[df["cmdDesc"] == commodity].copy()
df_cmd.head()

In [ ]:
top10_countries = (
    df_cmd.groupby("partnerDesc", as_index=False)["primaryValue"]
    .sum()
    .sort_values("primaryValue", ascending=False)
    .head(10)["partnerDesc"]
    .tolist()
)

top10_countries

In [ ]:
df_top = df_cmd[df_cmd["partnerDesc"].isin(top10_countries)].copy()
df_top.head()

In [ ]:
pivot_df = (
    df_top.groupby(["partnerDesc", "refYear"], as_index=False)["primaryValue"]
    .sum()
    .pivot(index="partnerDesc", columns="refYear", values="primaryValue")
    .fillna(0)
)

pivot_df

In [ ]:
pivot_df["Total"] = pivot_df.sum(axis=1)
pivot_df = pivot_df.sort_values("Total", ascending=False)
pivot_df = pivot_df.drop(columns="Total")

pivot_df

In [ ]:
def choose_usd_scale(max_value):
    if max_value >= 1e12:
        return 1e12, "Trillion USD"
    elif max_value >= 1e9:
        return 1e9, "Billion USD"
    elif max_value >= 1e6:
        return 1e6, "Million USD"
    else:
        return 1, "USD"

In [ ]:
countries = pivot_df.index.tolist()
years = pivot_df.columns.tolist()

# find the largest value in the chart
max_value = pivot_df.max().max()

# choose scale and y-axis label
scale_factor, y_label = choose_usd_scale(max_value)

x = np.arange(len(countries))
bar_width = 0.8 / len(years)

plt.figure(figsize=(14, 7))

for i, year in enumerate(years):
    plt.bar(
        x + i * bar_width,
        pivot_df[year] / scale_factor,
        width=bar_width,
        label=str(int(year))
    )

plt.xticks(
    x + bar_width * (len(years) - 1) / 2,
    countries,
    rotation=45,
    ha="right"
)

plt.ylabel(y_label)
plt.xlabel("Partner Country")
plt.title(f"Top 10 Partner Countries by Primary Value for {commodity}")
plt.legend(title="Year")

# remove scientific notation like 1e8
plt.ticklabel_format(style="plain", axis="y")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.ticker as mticker

In [ ]:
safe_name = commodity.replace("/", "_").replace("\\", "_").replace(";", "").replace(" ", "_")

countries = pivot_df.index.tolist()
years = pivot_df.columns.tolist()

max_value = pivot_df.max().max()
scale_factor, y_label = choose_usd_scale(max_value)

x = np.arange(len(countries))
bar_width = 0.8 / len(years)

plt.figure(figsize=(14, 7))

for i, year in enumerate(years):
    plt.bar(
        x + i * bar_width,
        pivot_df[year] / scale_factor,
        width=bar_width,
        label=str(int(year))
    )

plt.xticks(
    x + bar_width * (len(years) - 1) / 2,
    countries,
    rotation=45,
    ha="right"
)

plt.ylabel(y_label)
plt.xlabel("Partner Country")
plt.title(f"Top 10 Partner Countries by Primary Value for {commodity}")
plt.legend(title="Year")
plt.ticklabel_format(style="plain", axis="y")
plt.tight_layout()

save_path = f"{output_folder}/{safe_name}.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", save_path)

In [ ]:
for commodity in df["cmdDesc"].unique():
    df_cmd = df[df["cmdDesc"] == commodity].copy()

    top10_countries = (
        df_cmd.groupby("partnerDesc", as_index=False)["primaryValue"]
        .sum()
        .sort_values("primaryValue", ascending=False)
        .head(10)["partnerDesc"]
        .tolist()
    )

    df_top = df_cmd[df_cmd["partnerDesc"].isin(top10_countries)].copy()

    pivot_df = (
        df_top.groupby(["partnerDesc", "refYear"], as_index=False)["primaryValue"]
        .sum()
        .pivot(index="partnerDesc", columns="refYear", values="primaryValue")
        .fillna(0)
    )

    if pivot_df.empty:
        continue

    pivot_df["Total"] = pivot_df.sum(axis=1)
    pivot_df = pivot_df.sort_values("Total", ascending=False)
    pivot_df = pivot_df.drop(columns="Total")

    countries = pivot_df.index.tolist()
    years = pivot_df.columns.tolist()

    max_value = pivot_df.max().max()
    scale_factor, y_label = choose_usd_scale(max_value)

    x = np.arange(len(countries))
    bar_width = 0.8 / len(years)

    plt.figure(figsize=(14, 7))

    for i, year in enumerate(years):
        plt.bar(
            x + i * bar_width,
            pivot_df[year] / scale_factor,
            width=bar_width,
            label=str(int(year))
        )

    plt.xticks(
        x + bar_width * (len(years) - 1) / 2,
        countries,
        rotation=45,
        ha="right"
    )

    plt.ylabel(y_label)
    plt.xlabel("Partner Country")
    plt.title(f"Top 10 Partner Countries by Primary Value for {commodity}")
    plt.legend(title="Year")
    plt.ticklabel_format(style="plain", axis="y")
    plt.gca().yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.2f}"))
    plt.tight_layout()

    safe_name = commodity.replace("/", "_").replace("\\", "_").replace(";", "").replace(" ", "_")
    save_path = f"{output_folder}/{safe_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
import os
os.listdir(output_folder)

In [ ]:
import matplotlib.ticker as mticker

In [ ]:
# ----- country share (%) to World using partnerDesc == "World" -----

# 1) total primaryValue by commodity, year, and partner
tmp = (
    df.groupby(["cmdDesc", "refYear", "partnerDesc"], as_index=False)["primaryValue"]
    .sum()
)

# 2) get the reported World primaryValue
world_df = (
    tmp[tmp["partnerDesc"].str.strip().str.lower() == "world"]
    [["cmdDesc", "refYear", "primaryValue"]]
    .rename(columns={"primaryValue": "world_primaryValue"})
)

# 3) keep non-World countries only
country_share_table = tmp[
    tmp["partnerDesc"].str.strip().str.lower() != "world"
].copy()

# 4) merge World value and calculate share
country_share_table = country_share_table.merge(
    world_df,
    on=["cmdDesc", "refYear"],
    how="left"
)

country_share_table["country_share"] = (
    country_share_table["primaryValue"] / country_share_table["world_primaryValue"]
)

country_share_table["country_share_percent"] = country_share_table["country_share"] * 100

# 5) format table
country_share_table["refYear"] = country_share_table["refYear"].astype(int)
country_share_table["country_share_percent"] = country_share_table["country_share_percent"].round(4)

country_share_table = country_share_table.sort_values(
    ["cmdDesc", "refYear", "country_share_percent"],
    ascending=[True, True, False]
).reset_index(drop=True)

country_share_table[["cmdDesc", "refYear", "partnerDesc", "primaryValue", "world_primaryValue", "country_share_percent"]].head(20)

In [ ]:
# ----- non-PFE MACR by commodity and year using World row as denominator -----

pfe_countries = ["China", "Russia", "Iran", "North Korea", "Korea, North", "Russia Federation"]

tmp = (
    df.groupby(["cmdDesc", "refYear", "partnerDesc"], as_index=False)["primaryValue"]
    .sum()
)

# World denominator from the actual World row
world_df = (
    tmp[tmp["partnerDesc"].str.strip().str.lower() == "world"]
    [["cmdDesc", "refYear", "primaryValue"]]
    .rename(columns={"primaryValue": "world_primaryValue"})
)

# remove World row
tmp_non_world = tmp[
    tmp["partnerDesc"].str.strip().str.lower() != "world"
].copy()

# merge denominator
tmp_non_world = tmp_non_world.merge(
    world_df,
    on=["cmdDesc", "refYear"],
    how="left"
)

# country share relative to reported World
tmp_non_world["country_share"] = (
    tmp_non_world["primaryValue"] / tmp_non_world["world_primaryValue"]
)

# keep only non-PFE countries
non_pfe_cmd = tmp_non_world[
    ~tmp_non_world["partnerDesc"].isin(pfe_countries)
].copy()

# sum shares of all non-PFE countries
macr_cmd = (
    non_pfe_cmd.groupby(["cmdDesc", "refYear"], as_index=False)["country_share"]
    .sum()
    .rename(columns={"country_share": "non_PFE_MACR"})
)

macr_cmd["non_PFE_MACR_percent"] = macr_cmd["non_PFE_MACR"] * 100
macr_cmd["refYear"] = macr_cmd["refYear"].astype(int)

macr_cmd = macr_cmd.sort_values(["cmdDesc", "refYear"]).reset_index(drop=True)
macr_cmd.head(20)

In [ ]:
check_2020 = world_df[world_df["refYear"] == 2024].copy()
check_2020

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

for commodity in macr_cmd["cmdDesc"].unique():
    temp = macr_cmd[macr_cmd["cmdDesc"] == commodity].copy()
    temp = temp.sort_values("refYear")

    plt.figure(figsize=(8, 5))
    plt.plot(temp["refYear"], temp["non_PFE_MACR_percent"], marker="o", linewidth=2)

    plt.xticks(temp["refYear"])   # no decimals in years
    plt.gca().xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:.0f}"))
    plt.gca().yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.1f}"))

    plt.xlabel("Year")
    plt.ylabel("non-PFE MACR (%)")
    plt.title(f"non-PFE MACR by Year for {commodity}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for commodity in macr_cmd["cmdDesc"].unique():
    temp = macr_cmd[macr_cmd["cmdDesc"] == commodity].copy().sort_values("refYear")
    plt.plot(temp["refYear"], temp["non_PFE_MACR_percent"], marker="o", linewidth=2, label=commodity)

plt.gca().xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:.0f}"))
plt.gca().yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.1f}"))
plt.xlabel("Year")
plt.ylabel("non-PFE MACR (%)")
plt.title("non-PFE MACR by Commodity and Year")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()